# 激活函数是深度学习的核心零件之一，它决定了神经网络能不能学到“非线性关系”。没有它，再深的网络也只是一个线性模型。

1. 最根本的作用：引入非线性
神经网络的每一层本质是做一次线性变换：

text
y = W·x + b
如果每一层都只有这个，那不管叠多少层，最终等价于一个线性变换：

text
y = W2·(W1·x + b1) + b2 = (W2·W1)·x + (W2·b1 + b2)
也就是说，100 层线性层 = 1 层线性层，网络再深也没用。

激活函数的作用就是在每层线性变换后加一个非线性函数：

text
y = activation(W·x + b)
这样多层叠加后才能拟合任意复杂的函数（万能逼近定理）。

一句话：激活函数让神经网络从“只会画直线”变成“能画任意曲线”。

2. 直观理解
没有激活函数	有激活函数
只能拟合直线/平面	能拟合曲线、曲面
无法解决 XOR 问题	可以解决 XOR
深度毫无意义	深度才有意义
经典例子：XOR 问题。
单层线性模型无法分开 XOR 的两类点，但加一个隐藏层 + 激活函数就能轻松解决。

3. 常见激活函数
Sigmoid
text
σ(x) = 1 / (1 + e^(-x))
输出范围 (0, 1)

缺点：梯度消失严重，输出不是零中心

现在主要用于二分类输出层

Tanh
text
tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))
输出范围 (-1, 1)，零中心

仍有梯度消失问题

ReLU（最常用）
text
ReLU(x) = max(0, x)
计算简单，正区间梯度恒为 1，缓解梯度消失

缺点：负区间梯度为 0，可能出现“神经元死亡”

是 CNN / MLP 的默认选择

Leaky ReLU / PReLU
text
LeakyReLU(x) = max(αx, x)，α 很小，如 0.01
负区间也给一点梯度，缓解神经元死亡

GELU / SiLU（Swish）
平滑版 ReLU，Transformer 里常用

BERT、GPT 系列默认用 GELU

Softmax
不是逐元素激活，而是把向量变成概率分布

用于多分类输出层

4. 在 CNN 里的作用
以 CIFAR-10 的卷积网络为例：

python
nn.Conv2d(3, 32, 3)   # 线性卷积
nn.ReLU()             # 非线性激活
nn.Conv2d(32, 64, 3)  # 再卷积
nn.ReLU()
每个卷积后接 ReLU：

卷积负责提取特征（边缘、纹理）

ReLU 负责引入非线性，让特征组合更丰富

没有 ReLU，多层卷积等价于一次卷积

5. 输出层用什么
任务	输出层激活
二分类	Sigmoid
多分类	Softmax（或直接输出 logits + CrossEntropyLoss）
回归	无（恒等映射）
注意：PyTorch 的 nn.CrossEntropyLoss 内部已经包含 Softmax，所以模型输出层不要再加 Softmax。

6. 为什么激活函数要非线性且可导
非线性：否则深度无意义

可导：反向传播需要求梯度

梯度性质好：避免梯度消失/爆炸

ReLU 在 0 处不可导，但工程上直接取次梯度，效果很好。

7. 一句话总结
激活函数的作用是给神经网络引入非线性，让多层网络能拟合复杂函数；没有它，再深的网络也只是一个线性模型。最常用的是 ReLU，输出层根据任务用 Sigmoid / Softmax / 无激活。

# 次梯度是凸函数在不可导点处“能当梯度用”的斜率；ReLU 在 0 处的次微分是 [0, 1]，PyTorch 取 0，工程上完全够用，因为 0 点几乎不会精确出现，且次梯度下降本身有收敛保证。

不是。神经网络不是越深越好，深度只是提升表达能力的一种手段，但超过某个点后会带来一系列问题，甚至让效果变差。

1. 为什么“深”有用
从函数角度看，深网络能表达更复杂的函数：

浅网络：需要指数级宽的神经元才能拟合某些函数

深网络：用多项式级参数就能拟合同样的函数

这就是深度带来的表达效率。比如：

图像识别：浅层学边缘、纹理，深层学物体部件、整体

语言模型：浅层学词法，深层学句法、语义、推理

所以深度本身是有价值的。

2. 但深了会出什么问题
（1）梯度消失 / 梯度爆炸
反向传播时，梯度要逐层回传。

每层梯度乘一个小于 1 的数 → 越传越小 → 梯度消失，浅层学不动

每层梯度乘一个大于 1 的数 → 越传越大 → 梯度爆炸，训练发散

Sigmoid/Tanh 时代，超过 5~10 层就很难训。

（2）退化问题（Degradation）
理论上，更深的网络至少不应该比浅的差——多加几层恒等映射就行。
但实验发现：56 层比 20 层训练误差还大，这不是过拟合，是优化变难了。

（3）过拟合
参数越多、越深，越容易记住训练集噪声，泛化变差。
数据量不够时尤其明显。

（4）计算和显存成本
训练时间线性甚至超线性增长

显存占用增大，batch size 被迫减小

推理延迟变高

（5）优化困难
损失曲面更复杂，容易陷入坏局部极小

对学习率、初始化、归一化更敏感

3. 怎么解决“深”带来的问题
问题	解决方案
梯度消失/爆炸	ReLU、BatchNorm、LayerNorm、残差连接
退化问题	残差连接（ResNet）
过拟合	Dropout、权重衰减、数据增强、更多数据
优化困难	更好的初始化（He/Xavier）、Adam、warmup
计算成本	模型压缩、蒸馏、剪枝、量化
残差连接是关键突破：

text
y = F(x) + x
梯度可以走“捷径”直接回传，让 100+ 层甚至 1000 层网络可训练。

4. 深度和宽度的权衡
不是只有“加深”一条路：

方向	特点
# 加深	表达效率高，但难训练
# 加宽	容易训练，但参数效率低
加数据	通常比加深度更有效
# 加归纳偏置	如 CNN、Transformer，结构比深度更重要
实际中，深度、宽度、数据量、正则化要一起调，不是单纯堆深度。

5. 经验规律
在一定范围内，加深能提升效果

超过某个点，收益递减，甚至变差

ResNet 之后，100 层、1000 层也能训，但并不是越深越好

Transformer 也是几十层为主，GPT-3 有 96 层，不是无限深

小数据集上，深网络往往不如浅网络 + 好正则

大数据集上，深网络优势才明显

6. 一个直观类比
把神经网络比作做菜：

浅网络：只会炒、煮

深网络：会切、腌、炒、调、摆盘，能做的菜更复杂

但太深：步骤太多，中间任何一步出错，整道菜就毁了

而且需要更多食材（数据）和更好的厨房管理（归一化、残差）

关键不是步骤多，而是每步都有用、能配合。

7. 什么时候该加深
数据量足够大

任务足够复杂（图像、语言）

有残差、归一化等训练技巧

计算资源充足

已经解决了过拟合

什么时候不该加深
数据量小

任务简单

训练不稳定

计算资源有限

浅模型已经够用

8. 一句话总结
神经网络不是越深越好：深度能提升表达能力，但会带来梯度消失、退化、过拟合、计算成本等问题；残差连接、归一化等技术让深网络可训练，但实际效果取决于数据量、任务复杂度、正则化和计算资源，存在一个“最优深度”，超过后收益递减甚至变差。

# 这两个都是神经网络里的核心概念。宽度决定“每层有多少神经元”，正则化决定“怎么防止模型死记硬背”。下面分别说清楚。

一、宽度（Width）
1. 是什么
宽度指的是每一层神经元的数量，也就是每层输出的特征维度。

python
nn.Linear(784, 512)   # 这一层宽度 = 512
nn.Linear(512, 256)   # 这一层宽度 = 256
nn.Linear(256, 10)    # 这一层宽度 = 10
对 CNN 来说：

python
nn.Conv2d(3, 32, 3)    # 输出通道数 32 → 宽度 32
nn.Conv2d(32, 64, 3)   # 输出通道数 64 → 宽度 64
对 Transformer 来说：

d_model（隐藏维度）就是宽度，比如 768、4096

FFN 中间层维度也是宽度

2. 宽度有什么用
宽度越大，每层能表示的特征越多

能同时关注更多模式，表达能力更强

训练更容易（相比加深），梯度不容易消失

直观理解：

宽度 = 一层里有多少个“侦探”同时看数据

侦探越多，能发现的可疑线索越多

3. 宽度太大有什么问题
参数暴涨，显存和计算成本高

容易过拟合

# 收益递减：从 512 加到 1024 可能有提升，从 4096 加到 8192 可能没感觉

4. 深度 vs 宽度
深度	宽度
作用	层层抽象，组合特征	每层同时看更多特征
优点	表达效率高	容易训练
缺点	难训练、梯度问题	参数效率低
类比	流水线工序多	每道工序人手多
实际中两者要平衡，不是单堆一个。

二、正则化（Regularization）
1. 是什么
正则化是防止模型过拟合的一系列技术。
过拟合 = 模型在训练集上表现很好，但在新数据上表现差，因为它把训练集的噪声也记住了。

正则化的目标：

让模型学“通用规律”，而不是“死记硬背训练样本”。

2. 常见正则化方法
（1）L2 正则（权重衰减）
在损失里加一项：

text
L = 原损失 + λ·Σw²
惩罚大权重，让权重尽量小

模型更平滑，不容易对个别样本敏感

PyTorch 里：

python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
（2）L1 正则
text
L = 原损失 + λ·Σ|w|
让部分权重变成 0，产生稀疏性

可用于特征选择

（3）Dropout
训练时随机“关掉”一部分神经元：

python
nn.Dropout(p=0.5)
防止神经元之间过度依赖

相当于训练多个子网络再集成

推理时关闭

（4）数据增强
对训练样本做变换，增加多样性：

图像：翻转、裁剪、旋转、颜色抖动

文本：同义词替换、回译

这是最有效的正则化之一，因为它直接增加数据多样性。

（5）Early Stopping
验证集损失不再下降就停止训练，防止继续拟合噪声。

（6）BatchNorm / LayerNorm
虽然主要作用是稳定训练，但也有轻微正则化效果。

（7）DropBlock / Stochastic Depth
更高级的 dropout 变体，用于 CNN / Transformer。

3. 正则化强度怎么调
太弱 → 过拟合

太强 → 欠拟合（模型学不动）

常见做法：

先用小正则，观察训练/验证曲线

训练 loss 降但验证 loss 升 → 加大正则

训练 loss 都降不下去 → 减小正则

三、四者怎么一起调
因素	作用	调大后果	调小后果
深度	提升抽象能力	难训练、退化	表达不足
宽度	提升每层容量	参数多、过拟合	容量不足
数据量	提供泛化基础	成本高	过拟合
正则化	防过拟合	欠拟合	过拟合
经验顺序：

先保证数据量够，数据不够时加深度/宽度意义不大

选合适深度，有残差、归一化再加深

调宽度，让每层容量匹配任务

加正则化，根据验证集表现调整强度

反复迭代，不是一次调好

四、一句话总结
宽度 = 每层神经元数量，决定每层能同时表示多少特征

正则化 = 防止过拟合的技术（L2、Dropout、数据增强等），让模型学通用规律而非死记硬背

实际中深度、宽度、数据量、正则化要一起平衡，单堆任何一个都容易出问题

# 1. 一句话区分
集合	作用	模型见过吗	用来干什么
训练集	学参数	见过	更新权重
验证集	调超参数	没见过	选模型、调参、早停
测试集	最终评估	没见过	报告最终性能
# 2. 训练集（Training Set）
作用
更新模型参数（权重、偏置）

模型通过它计算损失、反向传播、梯度下降

特点
占数据大头，通常 60%~80%

模型会“记住”它，训练损失通常最低

训练集表现好 ≠ 模型好

类比
学生平时做的练习题，用来学习知识。

3. 验证集（Validation Set）
作用
调超参数：学习率、层数、宽度、正则化强度

选模型：多个模型/checkpoint 里挑最好的

早停：验证损失不再下降就停

检测过拟合：训练损失降、验证损失升 → 过拟合

特点
模型训练时不更新参数，但会被“看”到

占 10%~20%

反复用它调参，会间接“泄漏”信息，所以不能当最终评估

类比
模拟考试，用来检验学习效果、调整复习策略。

关键点
验证集是决策依据：

python
if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), "best.pth")
4. 测试集（Test Set）
作用
最终评估模型泛化能力

报告论文/项目的最终指标（准确率、F1 等）

特点
训练和调参时完全不能碰

只在最后用一次

占 10%~20%

如果反复用测试集调参，它就变成验证集了，评估不再公正

类比
高考，只考一次，成绩代表真实水平。

5. 为什么不能只用训练集和测试集
如果只有训练集 + 测试集：

调参时你只能看测试集

每调一次就看一次测试集 → 测试集信息泄漏

最终报告的性能偏乐观，不能反映真实泛化

所以需要验证集作为“调参专用”，测试集留到最后。

6. 三者关系图
text
全部数据
├── 训练集（60~80%）  → 学参数
├── 验证集（10~20%）  → 调超参、选模型、早停
└── 测试集（10~20%）  → 最终评估，只用一次
CIFAR-10 的例子：

text
官方 train=True  (50000)
├── train_set (45000)  → 训练
└── val_set   ( 5000)  → 验证

官方 train=False (10000)
└── test_set          → 测试
7. 常见划分比例
数据量	训练	验证	测试
小（<1万）	60%	20%	20%
中（1万~100万）	80%	10%	10%
大（>100万）	98%	1%	1%
数据越多，验证/测试可以越小，因为绝对数量已经够。

8. 训练过程中三者表现
现象	训练损失	验证损失	含义
正常	下降	下降	模型在学
过拟合	下降	上升	记住噪声了
欠拟合	高	高	模型太弱/没训够
收敛	平	平	可以停
测试集只在最后看一次，用来验证“验证集上选的模型”是否真的泛化好。

9. 常见误区
误区	问题
用测试集调参	测试集泄漏，评估不公正
没有验证集	只能靠测试集调参，同样泄漏
验证集太小	评估噪声大，选模型不准
训练集太小	模型学不够，欠拟合
三个集合分布不一致	评估无意义
10. 一句话总结
训练集：学参数，模型见过

验证集：调超参、选模型、早停，模型没学过但被“看”过

测试集：最终评估，只用一次，模型完全没见过

三者分工明确：训练集负责“学”，验证集负责“选”，测试集负责“考”。